# 24 — Quantization

**Topic:** Quantization  
**Audience:** AI PMs, founders, ML engineers, and portfolio builders.  
**How to use this notebook:** run the toy implementation first, then replace the toy data/model with your company data or project data.

## Mental model

This notebook is part of a 32-part LLM systems implementation path. Each notebook follows the same pattern:

1. Product intuition: what capability this unlocks.
2. Core idea: the minimum math or architecture you need.
3. Implementation from scratch or near-scratch.
4. Production notes: cost, risk, reliability, and founder decisions.
5. Portfolio extension: how to turn this into a public project.


In [ ]:
from pathlib import Path
import math, random, re, json, os, statistics, time, queue, threading
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except Exception as e:
    TfidfVectorizer = None
    cosine_similarity = None
    print('Optional sklearn import failed:', e)

# Make notebooks runnable from the pack root OR from a notebook subdirectory.
def find_pack_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for p in [start] + list(start.parents):
        if (p / 'data' / 'tiny_corpus.txt').exists():
            return p
    # Fallback for the sandbox path used to create this pack.
    sandbox = Path('/mnt/data/advanced_llm_systems_notebook_pack')
    if (sandbox / 'data' / 'tiny_corpus.txt').exists():
        return sandbox
    return start

ROOT = find_pack_root()
DATA = ROOT / 'data'
print('Pack root:', ROOT)
print('PyTorch:', torch.__version__)
torch.set_num_threads(1)
torch.manual_seed(7)
random.seed(7)
np.random.seed(7)

def basic_tokenize(text: str):
    return re.findall(r"[a-zA-Z]+|\d+|[^\w\s]", text.lower())

def count_params(model):
    return sum(p.numel() for p in model.parameters())

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B,T,C = x.shape
        q,k,v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k = k.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        v = v.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        att = q @ k.transpose(-2,-1) / math.sqrt(self.d_head)
        mask = torch.tril(torch.ones(T,T,device=x.device)).bool()
        att = att.masked_fill(~mask[None,None,:,:], float('-inf'))
        w = torch.softmax(att, dim=-1)
        y = self.dropout(w) @ v
        y = y.transpose(1,2).contiguous().view(B,T,C)
        return self.proj(y)

class TransformerBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4, mlp_ratio=4, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio*d_model), nn.GELU(),
            nn.Linear(mlp_ratio*d_model, d_model), nn.Dropout(dropout)
        )
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

# Shared toy RAG utilities for notebooks that need retrieval.
def load_rag_docs():
    path = DATA / 'rag_docs.jsonl'
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

docs = load_rag_docs()
_retriever_state = {}
def build_retriever(docs_in=None):
    docs_in = docs if docs_in is None else docs_in
    if not docs_in or TfidfVectorizer is None:
        return None, None
    vectorizer = TfidfVectorizer(stop_words='english')
    X = vectorizer.fit_transform([d['title'] + ' ' + d['text'] for d in docs_in])
    return vectorizer, X

_default_vectorizer, _default_X = build_retriever(docs)
def retrieve(query, k=2):
    if _default_vectorizer is None or _default_X is None:
        return []
    q = _default_vectorizer.transform([query])
    sims = cosine_similarity(q, _default_X)[0]
    idx = sims.argsort()[::-1][:k]
    return [{**docs[i], 'score': float(sims[i])} for i in idx]

# Shared eval helpers.
def exact_contains(output, expected):
    return str(expected).lower() in str(output).lower()

def valid_json_with_key(output, key):
    try:
        return key in json.loads(output)
    except Exception:
        return False

def run_eval(cases, system_fn):
    rows = []
    for c in cases:
        out = system_fn(c['input'])
        if 'expected' in c:
            passed = exact_contains(out, c['expected'])
        elif 'expected_key' in c:
            passed = valid_json_with_key(out, c['expected_key'])
        else:
            passed = False
        rows.append({**c, 'output': out, 'passed': passed})
    return pd.DataFrame(rows)


## Why this matters

Quantization reduces model memory and can speed inference by using lower-precision weights/activations. It is central to serving cost and edge deployment.

This notebook implements simple symmetric int8 and group-wise int4 quantization for teaching.


In [ ]:
W = torch.randn(128, 64) * 0.2

def quantize_int8_symmetric(W):
    scale = W.abs().max() / 127
    q = torch.clamp(torch.round(W / scale), -127, 127).to(torch.int8)
    return q, scale

def dequantize_int8(q, scale):
    return q.float() * scale

q, scale = quantize_int8_symmetric(W)
W_hat = dequantize_int8(q, scale)
print('MAE:', float((W-W_hat).abs().mean()), 'max error:', float((W-W_hat).abs().max()))


In [ ]:
def quantize_groupwise_int4(W, group_size=32):
    flat = W.flatten()
    q_vals, scales = [], []
    for start in range(0, flat.numel(), group_size):
        chunk = flat[start:start+group_size]
        scale = chunk.abs().max() / 7 if chunk.abs().max() > 0 else torch.tensor(1.0)
        q = torch.clamp(torch.round(chunk / scale), -8, 7).to(torch.int8)
        q_vals.append(q); scales.append(scale)
    return torch.cat(q_vals).view_as(W), torch.stack(scales), W.shape

def dequantize_groupwise_int4(q, scales, shape, group_size=32):
    flat = q.flatten().float()
    outs = []
    for i,start in enumerate(range(0, flat.numel(), group_size)):
        outs.append(flat[start:start+group_size] * scales[i])
    return torch.cat(outs).view(shape)

q4, scales4, shape = quantize_groupwise_int4(W)
W4 = dequantize_groupwise_int4(q4, scales4, shape)
print('int4-ish MAE:', float((W-W4).abs().mean()))


In [ ]:
x = torch.randn(16, 64)
y = x @ W.T
y8 = x @ W_hat.T
y4 = x @ W4.T
print('output int8 MAE:', float((y-y8).abs().mean()))
print('output int4 MAE:', float((y-y4).abs().mean()))


## Founder notes

- Quantization can reduce cost, but always evaluate quality on your task.
- Weight-only quantization helps memory; activation/KV quantization can help further but adds risk.
- QLoRA combines low-bit base weights with trainable adapters.
- Some serving stacks support FP8/INT4 differently depending on GPU generation.


## Portfolio / company reuse checklist

To reuse this notebook in a real project:

- Replace the toy corpus/data with a small but representative sample from your domain.
- Add a held-out evaluation set before optimizing anything.
- Track failure cases by category, not just aggregate accuracy/loss.
- Decide whether this is a prototype-only component, a production component, or a learning artifact.
- Document assumptions in a model card or system card.


## References to review after implementation

Use the implementation above first, then read the original papers/docs to connect the code to production systems. The README contains a consolidated reference list for the full pack.
